# 01 — Data validation

Checks the cleaned hourly dataset produced by `python -m src.data.clean` before any analysis:

1. coverage and missing values by column and period
2. duplicate timestamps and DST day lengths (23 / 25 hours)
3. which load-forecast vintage was used (look-ahead check)
4. value ranges and the most extreme hours

The 2026 final holdout (`hourly_holdout.parquet`) is deliberately **not** loaded here beyond its row count.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # project root when run from notebooks/
import pandas as pd
from src.config import load_config
from src.data.clean import validate

pd.set_option("display.width", 140)
cfg = load_config()
proc = cfg["paths"]["processed"]
df = pd.read_parquet(proc / "hourly.parquet")
holdout_rows = len(pd.read_parquet(proc / "hourly_holdout.parquet", columns=["timestamp_utc"]))
print(f"research rows: {len(df):,} | holdout rows (not inspected): {holdout_rows:,}")
print(df["timestamp_ept"].iloc[0], "->", df["timestamp_ept"].iloc[-1])
df.head()

## 1. Coverage by period

In [ ]:
value_cols = ["da_lmp", "rt_lmp", "forecast_load", "actual_load"]
coverage = df.groupby("period", sort=False).agg(
    first_day=("market_date", "min"), last_day=("market_date", "max"), hours=("timestamp_utc", "size"),
    **{f"missing_{c}": (c, lambda s: int(s.isna().sum())) for c in value_cols},
)
coverage

In [ ]:
missing = df.loc[df[value_cols].isna().any(axis=1), ["timestamp_ept", "period", *value_cols]]
print(f"hours with any missing value: {len(missing)}")
missing.head(20)

## 2. Duplicates and DST day lengths

In [ ]:
rep = validate(df)
print("duplicate timestamps:", rep.duplicate_timestamps)
print("days whose length is not explained by DST:", rep.unexpected_day_lengths or "none")
pd.Series(rep.dst_days, name="hours").rename_axis("market_date").to_frame()

In [ ]:
# fall-back days: the repeated HE2 should appear exactly once with is_dst_repeat=True
df.loc[df["hours_in_day"] == 25, ["timestamp_utc", "timestamp_ept", "hour_ending", "is_dst_repeat", "da_lmp", "rt_lmp"]].iloc[:4]

## 3. Load-forecast vintage (look-ahead check)

Every forecast used must have been evaluated **before 10:30 EPT on D−1**. Expected: almost all from the 09:45 EPT vintage, lead time ≈ 14–38 hours.

In [ ]:
ev_ept = df["forecast_evaluated_utc"].dt.tz_convert("America/New_York")
vint = pd.DataFrame({
    "evaluated_time_ept": ev_ept.dt.strftime("%H:%M"),
    "days_before_operating_day": (df["market_date"] - ev_ept.dt.tz_localize(None).dt.normalize()).dt.days,
})
print(vint.value_counts(dropna=False).rename("hours").to_frame())
print()
print(df["forecast_lead_hours"].describe().round(1))

In [ ]:
decision = (df["market_date"] - pd.Timedelta(days=1) + pd.Timedelta(hours=10, minutes=30)).dt.tz_localize("America/New_York")
violations = (df["forecast_evaluated_utc"] > decision.dt.tz_convert("UTC")).sum()
print("forecasts evaluated after the decision time:", int(violations))
assert violations == 0

## 4. Value ranges and extremes

In [ ]:
df[["da_lmp", "rt_lmp", "da_rt_spread", "forecast_load", "actual_load", "load_forecast_error"]].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(1)

In [ ]:
print("warnings:", rep.warnings or "none")
cols = ["timestamp_ept", "da_lmp", "rt_lmp", "da_rt_spread", "actual_load", "load_forecast_error"]
print()
print("Largest positive spreads (RT >> DA)")
display(df.nlargest(5, "da_rt_spread")[cols])
print("Largest negative spreads (RT << DA)")
display(df.nsmallest(5, "da_rt_spread")[cols])

## Summary

* **Coverage:** 27,048 research hours (Dec 2022 lookback → Dec 2025) plus 7,295 holdout hours (Jan–Oct 2026, not inspected). DA LMP, RT LMP and metered load are **complete** — no missing hours.
* **Duplicates / DST:** no duplicate timestamps; every spring-forward day has 23 hours and every fall-back day 25, with the repeated HE2 flagged by `is_dst_repeat`. PJM's EPT labels agree with our UTC→EPT conversion on every row.
* **Load forecast:** 192 research hours (0.7%) have no forecast available at decision time. These are gaps in PJM's publication, not pipeline errors:
  * spring-forward days 2024-03-10 and 2025-03-09 — no D−1 morning forecast was published;
  * the last hour of each fall-back day — the D−1 09:45 forecast has 24 rows for a 25-hour day;
  * 2023-03-11, 2023-12-09/10, 2023-12-25, 2024-01-21, 2024-06-29 — no D−1 forecast published.
  They are left as missing (filling them would require a later forecast, i.e. look-ahead). Forecast-based analyses drop these hours.
* **Look-ahead check:** 0 forecasts were evaluated after 10:30 EPT on D−1. 99.9% use the 09:45 D−1 vintage; 23 hours (2023-03-12) fall back to the 23:45 D−2 vintage. Lead time 14–46 h.
* **Extremes:** the largest positive spreads are Winter Storm Elliott (23–24 Dec 2022, RT up to $3,794/MWh vs DA ≈ $206, load ≈ 12 GW above forecast) — inside the lookback month, so excluded from research statistics but relevant for any rolling feature computed in early January 2023. The largest negative spreads are 25 Dec 2022 and 29 Jul 2025 (DA $550 vs RT $232).
* First look at 2023–2025 (excluding lookback): mean spread −$0.95/MWh, median −$2.78, std $29.4, range −$319 to +$1,477 — RT usually clears slightly below DA, with a long positive tail. Studied properly in Phase 2.